# Sphere-Void Model (Coupled Neutron-Photon) - FENDL-3.2c

This notebook compares YAMC and OpenMC simulation results for a sphere with a cylindrical void,
using coupled neutron-photon transport with the FENDL-3.2c nuclear data library.

## Model Description

- **Geometry**: 35 cm radius sphere with a 1 cm radius cylindrical void along the z-axis, vacuum boundaries
- **Source**: Point source at origin (inside void), 14.06 MeV monoenergetic neutrons, isotropic
- **Material**: Single nuclide at 1.0 g/cm³, temperature 294 K
- **Transport**: Coupled neutron-photon (neutrons produce secondary photons)
- **Statistics**: 500,000 particles × 10 batches = 5,000,000 total histories
- **Tallies**: Photon flux (energy-binned, VITAMIN-J-42), coherent-scatter, incoherent-scatter, photoelectric, pair-production

In [ ]:
# Setup and imports
import numpy as np
import pandas as pd
import glob
import os
import yamc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.0": "/home/jon/nuclear_data/endf-b8.0-hdf5/",
    "tendl-2021": "/home/jon/nuclear_data/tendl-2021-hdf5/tendl-2021-hdf5/",
    "fendl-3.2c": "/home/jon/nuclear_data/fendl-3.2c-hdf5/",
}

NUCLEAR_DATA = "fendl-3.2c"
PARTICLES = 500000
BATCHES = 10
SOURCE_ENERGY = 14.06e6  # 14.06 MeV neutrons
MAX_PLOT_ENERGY = 20e6  # 20 MeV (photon energies from neutron capture can be high)
FLUX_DECADES = 6

# Geometry dimensions
SPHERE_RADIUS = 35.0  # cm
CYLINDER_RADIUS = 1.0  # cm (void channel)
CYLINDER_HALF_HEIGHT = SPHERE_RADIUS  # extend cylinder through entire sphere

# Photon energy group structure
PHOTON_GROUP_STRUCTURE = 'VITAMIN-J-42'

In [ ]:
def run_openmc_simulation(nuclide, nuclear_data):
    """Run OpenMC coupled neutron-photon simulation for a single nuclide."""
    import openmc as mc

    mc.reset_auto_ids()
    mc.config['cross_sections'] = f"{NUCLEAR_DATA_DIR[nuclear_data]}cross_sections.xml"

    # Geometry: sphere with cylindrical void
    sphere = mc.Sphere(r=SPHERE_RADIUS, boundary_type='vacuum')
    cylinder = mc.ZCylinder(r=CYLINDER_RADIUS)
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT)
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT)

    # Material region: inside sphere, outside cylinder
    material_region = -sphere & (+cylinder | +z_top | -z_bot)

    # Void region: inside cylinder and inside sphere
    void_region = -sphere & -cylinder & +z_bot & -z_top

    # Material
    material = mc.Material()
    material.add_nuclide(nuclide, 1.0)
    material.set_density("g/cm3", 1.0)
    material.temperature = 294

    material_cell = mc.Cell(name="material", region=material_region, fill=material)
    void_cell = mc.Cell(name="void", region=void_region)
    geometry = mc.Geometry([material_cell, void_cell])

    # Source: 14.06 MeV isotropic neutrons at origin
    source = mc.IndependentSource(
        space=mc.stats.Point([0, 0, 0]),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        angle=mc.stats.Isotropic(),
    )
    source.particle = "neutron"

    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        photon_transport=True,
        output={'tallies': False, 'summary': False},
    )

    # Filters and tallies - photon tallies in the material cell
    cell_filter = mc.CellFilter(material_cell)
    photon_filter = mc.ParticleFilter("photon")
    energy_filter = mc.EnergyFilter.from_group_structure(PHOTON_GROUP_STRUCTURE)
    energy_bins_2d = energy_filter.bins
    energy_bins = np.concatenate([[energy_bins_2d[0, 0]], energy_bins_2d[:, 1]])

    tallies = []

    # Photon interaction score tallies
    photon_scores = ['coherent-scatter', 'incoherent-scatter', 'photoelectric', 'pair-production']
    for score in ['flux'] + photon_scores:
        tally = mc.Tally()
        tally.filters = [cell_filter, photon_filter]
        tally.scores = [score]
        tally.name = score
        tallies.append(tally)

    # Energy-binned photon flux tally
    tally_spectrum = mc.Tally()
    tally_spectrum.filters = [cell_filter, photon_filter, energy_filter]
    tally_spectrum.scores = ['flux']
    tally_spectrum.name = "flux_energy_binned"
    tallies.append(tally_spectrum)

    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    sp_filename = model.run(apply_tally_results=True)
    sp = mc.StatePoint(sp_filename)

    transport_time = sp.runtime['transport']
    total_particles = PARTICLES * BATCHES
    particles_per_second = total_particles / transport_time

    results = {
        'nuclide': nuclide,
        'code': 'openmc',
        'particles/s': particles_per_second,
        'tallies': {},
        'tally_std': {},
    }

    for tally in tallies:
        if tally.name != "flux_energy_binned":
            results['tallies'][tally.name] = tally.mean.flatten()[0]
            results['tally_std'][tally.name] = tally.std_dev.flatten()[0]

    results['flux_spectra_mean'] = np.array(tally_spectrum.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(tally_spectrum.std_dev).squeeze()
    results['energy_bins'] = energy_bins

    return results

In [ ]:
def run_yamc_simulation(nuclide, nuclear_data):
    """Run YAMC coupled neutron-photon simulation for a single nuclide. Returns (results_dict, model)."""
    import yamc as mc

    # Determine element from nuclide name (e.g., 'Fe56' -> 'Fe')
    element = ''.join(c for c in nuclide if not c.isdigit())

    neutron_path = f"{NUCLEAR_DATA_DIR[nuclear_data]}neutron/{nuclide}.h5"
    photon_path = f"{NUCLEAR_DATA_DIR[nuclear_data]}photon/{element}.h5"

    # Geometry: sphere with cylindrical void
    sphere = mc.Sphere(r=SPHERE_RADIUS, boundary_type='vacuum')
    cylinder = mc.ZCylinder(r=CYLINDER_RADIUS)
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT)
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT)

    # Material region: inside sphere, outside cylinder
    material_region = -sphere & (+cylinder | +z_top | -z_bot)

    # Void region: inside cylinder and inside sphere
    void_region = -sphere & -cylinder & +z_bot & -z_top

    # Material - coupled neutron-photon data
    material = mc.Material()
    material.material_id = 1
    material.add_nuclide(nuclide, 1.0)
    material.set_density("g/cm3", 1.0)
    material.temperature = 294
    material.read_nuclear_data(
        {nuclide: neutron_path},
        photon_data={element: photon_path},
    )

    material_cell = mc.Cell(cell_id=1, name="material", region=material_region, fill=material)
    void_cell = mc.Cell(cell_id=2, name="void", region=void_region)
    geometry = mc.Geometry([material_cell, void_cell])

    # Source: 14.06 MeV isotropic neutrons at origin
    source = mc.IndependentSource(
        space=mc.stats.Point([0, 0, 0]),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        angle=mc.stats.Isotropic(),
        particle_type="neutron",
    )

    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        photon_transport=True,
    )

    # Filters and tallies - photon tallies in the material cell
    cell_filter = mc.CellFilter(material_cell)
    photon_filter = mc.ParticleFilter("photon")
    energy_filter = mc.EnergyFilter.from_group_structure(PHOTON_GROUP_STRUCTURE)
    energy_bins = np.array(energy_filter.bins)

    tallies = []

    # Photon interaction score tallies
    photon_scores = ['coherent-scatter', 'incoherent-scatter', 'photoelectric', 'pair-production']
    for score in ['flux'] + photon_scores:
        tally = mc.Tally()
        tally.filters = [cell_filter, photon_filter]
        tally.scores = [score]
        tally.name = score
        tallies.append(tally)

    # Energy-binned photon flux tally
    tally_spectrum = mc.Tally()
    tally_spectrum.filters = [cell_filter, photon_filter, energy_filter]
    tally_spectrum.scores = ['flux']
    tally_spectrum.name = "flux_energy_binned"
    tallies.append(tally_spectrum)

    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    model.run()

    results = {
        'nuclide': nuclide,
        'code': 'yamc',
        'particles/s': model.particles_per_second,
        'tallies': {},
        'tally_std': {},
    }

    for tally in tallies:
        if tally.name != "flux_energy_binned":
            results['tallies'][tally.name] = tally.mean[0]
            results['tally_std'][tally.name] = tally.std_dev[0]

    results['flux_spectra_mean'] = np.array(tally_spectrum.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(tally_spectrum.std_dev).squeeze()
    results['energy_bins'] = energy_bins

    return results, model

In [ ]:
def create_step_trace(energy_bins, values, name, color, dash=None):
    """Create a step plot trace for Plotly (similar to matplotlib step with where='post')."""
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])

    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
    )


def create_comparison_plot(openmc_results, yamc_results):
    """Create a Plotly subplot with photon flux spectra (top) and ratio (bottom)."""

    energy_bins = openmc_results['energy_bins']
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])

    openmc_flux = openmc_results['flux_spectra_mean'].copy()
    openmc_std = openmc_results['flux_spectra_std_dev'].copy()
    yamc_flux = yamc_results['flux_spectra_mean'].copy()
    yamc_std = yamc_results['flux_spectra_std_dev'].copy()

    # Find the lowest energy bin with non-zero flux in either code
    has_flux = (openmc_flux > 0) | (yamc_flux > 0)
    if np.any(has_flux):
        first_nonzero_idx = np.argmax(has_flux)
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]

    max_flux = max(
        np.max(openmc_flux[openmc_flux > 0]) if np.any(openmc_flux > 0) else 1e-10,
        np.max(yamc_flux[yamc_flux > 0]) if np.any(yamc_flux > 0) else 1e-10
    )
    flux_threshold = max_flux / (10 ** FLUX_DECADES)

    openmc_flux_plot = np.where(openmc_flux >= flux_threshold, openmc_flux, np.nan)
    yamc_flux_plot = np.where(yamc_flux >= flux_threshold, yamc_flux, np.nan)
    openmc_std_plot = np.where(openmc_flux >= flux_threshold, openmc_std, np.nan)
    yamc_std_plot = np.where(yamc_flux >= flux_threshold, yamc_std, np.nan)

    # Calculate ratio with error propagation
    valid = (openmc_flux > 0) & (yamc_flux > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(valid, yamc_flux / openmc_flux, np.nan)
        rel_err_yamc = np.where(yamc_flux > 0, yamc_std / yamc_flux, 0)
        rel_err_openmc = np.where(openmc_flux > 0, openmc_std / openmc_flux, 0)
    ratio_rel_err = np.sqrt(rel_err_yamc**2 + rel_err_openmc**2)
    ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=('Photon Flux Spectra', 'YAMC / OpenMC Ratio')
    )

    fig.add_trace(
        create_step_trace(energy_bins, openmc_flux_plot, 'OpenMC', 'blue'),
        row=1, col=1
    )
    fig.add_trace(
        create_step_trace(energy_bins, yamc_flux_plot, 'YAMC', 'red', dash='dash'),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=openmc_flux_plot, mode='markers',
            name='OpenMC error',
            marker=dict(color='blue', size=4, symbol='circle'),
            error_y=dict(type='data', array=openmc_std_plot, visible=True, color='blue', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=yamc_flux_plot, mode='markers',
            name='YAMC error',
            marker=dict(color='red', size=4, symbol='square'),
            error_y=dict(type='data', array=yamc_std_plot, visible=True, color='red', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )

    fig.add_trace(
        create_step_trace(energy_bins, ratio, 'YAMC/OpenMC', 'green'),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([bin_centers, bin_centers[::-1]]),
            y=np.concatenate([ratio + ratio_std, (ratio - ratio_std)[::-1]]),
            fill='toself',
            fillcolor='rgba(0, 128, 0, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            name='Uncertainty',
            showlegend=False,
        ),
        row=2, col=1
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

    fig.update_xaxes(type="log", title_text="Energy (eV)",
                     range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
                     exponentformat='e', row=2, col=1)
    fig.update_xaxes(type="log",
                     range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
                     exponentformat='e', row=1, col=1)
    fig.update_yaxes(type="log", title_text="Photon Flux", exponentformat='e', row=1, col=1)
    fig.update_yaxes(title_text="Ratio", range=[0.5, 1.5], row=2, col=1)

    fig.update_layout(
        height=700,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=60, r=20, t=40, b=40)
    )

    return fig


def calculate_spectral_agreement(openmc_results, yamc_results, flux_threshold=1e-6):
    """Calculate statistical agreement metrics for photon flux spectra comparison."""

    openmc_flux = openmc_results['flux_spectra_mean']
    openmc_std = openmc_results['flux_spectra_std_dev']
    yamc_flux = yamc_results['flux_spectra_mean']
    yamc_std = yamc_results['flux_spectra_std_dev']

    chi_squared_terms = []
    rel_diffs = []
    weighted_rel_diffs = []
    flux_weights = []

    for i in range(len(openmc_flux)):
        omc = openmc_flux[i]
        ymc = yamc_flux[i]
        omc_std = openmc_std[i]
        ymc_std = yamc_std[i]

        combined_std = np.sqrt(omc_std**2 + ymc_std**2)
        if combined_std > 0:
            chi_squared_terms.append(((ymc - omc) / combined_std)**2)

        if omc > flux_threshold or ymc > flux_threshold:
            if omc > 0:
                rel_diff = abs((ymc - omc) / omc) * 100
                if np.isfinite(rel_diff):
                    rel_diffs.append(rel_diff)
                    weight = max(omc, ymc)
                    weighted_rel_diffs.append(rel_diff * weight)
                    flux_weights.append(weight)

    chi_squared = np.sum(chi_squared_terms)
    dof = len(chi_squared_terms)
    reduced_chi_squared = chi_squared / dof if dof > 0 else 0.0

    median_rel_diff = np.median(rel_diffs) if rel_diffs else 0.0
    weighted_avg_rel_diff = np.sum(weighted_rel_diffs) / np.sum(flux_weights) if flux_weights else 0.0

    if reduced_chi_squared < 1.5:
        quality = "Excellent"
    elif reduced_chi_squared < 3.0:
        quality = "Good"
    elif reduced_chi_squared < 5.0:
        quality = "Fair"
    else:
        quality = "Poor"

    return {
        'reduced_chi_squared': reduced_chi_squared,
        'degrees_of_freedom': dof,
        'median_rel_diff': median_rel_diff,
        'weighted_avg_rel_diff': weighted_avg_rel_diff,
        'quality': quality,
        'bins_analyzed': len(rel_diffs),
        'total_bins': len(openmc_flux)
    }


def create_agreement_table(stats):
    """Create a summary table of spectral agreement statistics."""
    rows = [
        {'Metric': 'Reduced Chi-Squared', 'Value': f"{stats['reduced_chi_squared']:.2f}",
         'Note': f"(ideal ~ 1.0, dof={stats['degrees_of_freedom']})"},
        {'Metric': 'Weighted Avg Rel. Diff', 'Value': f"{stats['weighted_avg_rel_diff']:.2f}%",
         'Note': '(flux-weighted)'},
        {'Metric': 'Median Rel. Diff', 'Value': f"{stats['median_rel_diff']:.2f}%",
         'Note': '(robust to outliers)'},
        {'Metric': 'Agreement Quality', 'Value': stats['quality'],
         'Note': f"({stats['bins_analyzed']}/{stats['total_bins']} bins analyzed)"},
    ]
    return pd.DataFrame(rows)


# Photon interaction MT descriptions
PHOTON_MT_DESCRIPTIONS = {
    'flux': 'Total photon flux',
    'coherent-scatter': 'Coherent (Rayleigh) scatter (MT 502)',
    'incoherent-scatter': 'Incoherent (Compton) scatter (MT 504)',
    'photoelectric': 'Photoelectric absorption (MT 522)',
    'pair-production': 'Pair production (MT 516)',
}


def create_tally_table(openmc_results, yamc_results):
    """Create a comparison table of photon tallies."""

    all_tallies = set(openmc_results['tallies'].keys()) | set(yamc_results['tallies'].keys())
    rows = []

    for name in sorted(all_tallies):
        openmc_val = openmc_results['tallies'].get(name, np.nan)
        openmc_std = openmc_results['tally_std'].get(name, np.nan)
        yamc_val = yamc_results['tallies'].get(name, np.nan)
        yamc_std = yamc_results['tally_std'].get(name, np.nan)

        if (openmc_val == 0 or np.isnan(openmc_val)) and (yamc_val == 0 or np.isnan(yamc_val)):
            continue

        if openmc_val != 0 and not np.isnan(openmc_val):
            rel_diff = (yamc_val - openmc_val) / openmc_val * 100
        else:
            rel_diff = np.nan

        reaction = PHOTON_MT_DESCRIPTIONS.get(name, name)

        rows.append({
            'Score': name,
            'Reaction': reaction,
            'OpenMC': f"{openmc_val:.4e} +/- {openmc_std:.2e}",
            'YAMC': f"{yamc_val:.4e} +/- {yamc_std:.2e}",
            'Rel. Diff (%)': f"{rel_diff:.2f}%" if not np.isnan(rel_diff) else 'N/A',
        })

    return pd.DataFrame(rows)

In [ ]:
# Get list of nuclides (need both neutron and photon data)
h5_files = sorted(glob.glob(os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron', '*.h5')))
nuclides = [os.path.splitext(os.path.basename(f))[0] for f in h5_files]

# Filter to nuclides that also have photon data for their element
photon_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'photon')
available_elements = set(
    os.path.splitext(os.path.basename(f))[0]
    for f in glob.glob(os.path.join(photon_dir, '*.h5'))
)

nuclides = [
    n for n in nuclides
    if ''.join(c for c in n if not c.isdigit()) in available_elements
]

if not nuclides:
    raise FileNotFoundError(
        f"No nuclides found with both neutron and photon data in {NUCLEAR_DATA_DIR[NUCLEAR_DATA]}"
    )

# For testing, limit to first few nuclides
# Comment out the next line to run all nuclides
nuclides = nuclides[:2]

print(f"Running coupled neutron-photon simulations for {len(nuclides)} nuclides...")
for nuclide in nuclides:
    print(f"  {nuclide}")

In [ ]:
# Run all simulations and collect results
all_results = {}
first_yamc_model = None

for nuclide in nuclides:
    print(f"\nProcessing {nuclide}...")
    yamc_results, yamc_model = run_yamc_simulation(nuclide, NUCLEAR_DATA)
    if first_yamc_model is None:
        first_yamc_model = yamc_model

    openmc_results = run_openmc_simulation(nuclide, NUCLEAR_DATA)
    all_results[nuclide] = {
        'openmc': openmc_results,
        'yamc': yamc_results
    }
    print(f"  OpenMC: {openmc_results['particles/s']:.0f} particles/s")
    print(f"  YAMC:   {yamc_results['particles/s']:.0f} particles/s")

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")

In [ ]:
# Display sphere-void model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", width=(80, 80), n_samples=500, axis_units="cm")
    with open("sphere_void_model_plot.html", "w") as f:
        f.write(html_plot)
    encoded = base64.b64encode(html_plot.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- Interactive photon flux spectra plot (log-log scale)
- YAMC/OpenMC ratio plot with uncertainty band
- Photon tally comparison table (flux, coherent-scatter, incoherent-scatter, photoelectric, pair-production)
- Performance comparison (particles/second)

In [ ]:
# Display results for each nuclide
for nuclide, results in all_results.items():
    openmc_results = results['openmc']
    yamc_results = results['yamc']

    # Header
    display(Markdown(f"## {nuclide}"))

    if openmc_results is None:
        display(Markdown(f"**OpenMC failed for this nuclide** (known issue with FENDL photon transport in OpenMC). Showing YAMC results only."))
        display(Markdown(f"**YAMC Performance:** {yamc_results['particles/s']:,.0f} particles/s"))

        # Show YAMC-only flux spectra
        energy_bins = yamc_results['energy_bins']
        yamc_flux = yamc_results['flux_spectra_mean']
        has_flux = yamc_flux > 0
        if np.any(has_flux):
            fig = go.Figure()
            fig.add_trace(create_step_trace(energy_bins, yamc_flux, 'YAMC', 'red'))
            fig.update_xaxes(type="log", title_text="Energy (eV)", exponentformat='e')
            fig.update_yaxes(type="log", title_text="Photon Flux", exponentformat='e')
            fig.update_layout(height=500, title=f"{nuclide} - Photon Flux Spectra (YAMC only)")
            fig.show()

        # Show YAMC-only tally table
        display(Markdown("### Photon Tallies (YAMC only)"))
        rows = []
        for name in sorted(yamc_results['tallies'].keys()):
            val = yamc_results['tallies'][name]
            std = yamc_results['tally_std'][name]
            if val == 0:
                continue
            reaction = PHOTON_MT_DESCRIPTIONS.get(name, name)
            rows.append({'Score': name, 'Reaction': reaction, 'YAMC': f"{val:.4e} +/- {std:.2e}"})
        if rows:
            display(pd.DataFrame(rows).style.hide(axis='index'))
    else:
        # Performance comparison
        openmc_pps = openmc_results['particles/s']
        yamc_pps = yamc_results['particles/s']
        speedup = yamc_pps / openmc_pps
        display(Markdown(f"**Performance:** OpenMC: {openmc_pps:,.0f} particles/s | YAMC: {yamc_pps:,.0f} particles/s | Speedup: {speedup:.2f}x"))

        # Flux spectra plot
        fig = create_comparison_plot(openmc_results, yamc_results)
        fig.update_layout(title=f"{nuclide} - Photon Flux Spectra (from coupled n,\u03b3 transport)")
        fig.show()

        # Spectral agreement statistics
        display(Markdown("### Spectral Agreement"))
        stats = calculate_spectral_agreement(openmc_results, yamc_results)
        agreement_table = create_agreement_table(stats)
        display(agreement_table.style.hide(axis='index'))

        # Tally table
        display(Markdown("### Photon Tally Comparison"))
        table = create_tally_table(openmc_results, yamc_results)
        display(table.style.hide(axis='index'))

    print()  # Add spacing between nuclides